In this lab, we will perform tasks where we will use a Python script as a controller - automater to a simple optimization process. 
The generic Python modules and procedures are not very efficient when it comes to serious number crunching. That's why many engineering libraries are written in C/C++/Fortan and often dedicated software, we want to use already exists. For this reason we wish to use a simple Python script as "duct tape" to the existing, hopefully efficient application. Today's exercise is based on this concept. 

Consider a Black-Box program that evaluates some interesting quantity. The computational process employed to compute this quantity could be very complex and time consuming. It could require the computations to be performed on an remote HPC cluster, but for now we assume that Black-Box accepts takes some problem parameters  and returns the computed output.

Arguments for the black box are defined in the setting file, that you can find in the current working directory, have a look.

Using Python:  
1. We will change the input parameters defined in the setting file. The setting file will be passed as an argument to the black box.
2. We will run the Black Box using Python. We can stop the Black box execution midway if desired.
3. We will read the output of the program using Python.
4. Design and implement a simple optimization strategy to find the minimum value of of the quantity returned by Black-Box

# Task 1

1.1 First, build your own executable Black-Box. You can use any building process you want. Using bash shell, once inside the `blacb_box` directory, this would be

```bash
mkdir build  #Create build Directory for isolating the executables from the source code
cd build     #Go to build directory
cmake ../    # This command will find your system configuration and save it to the build folder
make         # Last 
```
If all is fine, you should have created the `black-box` executable.

1.2 Try to run the `black-box` executable using the setting file that you can find in the main laboratory 4 directory.

```bash
black_box/bild/black-box --sett settings  
```

You should see something like this:
```
===== Starting process =====
X:	33
Y:	543
Z:	249027
```
Note that a `data.dat` file has been created, which contains all the data you will need.

Alternativly, you can try to build Black-Box from this notebook, by executing bash commands using `os.sytem`.

In [ ]:
import os
os.system('mkdir black_box/build')
os.chdir('./black_box/build')
os.system('cmake ../')
os.system('make')
os.chdir('../../')

In [ ]:
pwd

# Task 2
Use `os.system` to call the black-box executable from within the python script. Than examine the `data.dat`

# Task 3
Create a Python function that changes parameters $(X, Y)$ in the setting file. The function should accept those two parameters and simply rewrite the file.

# Task 4
Write a function for parsing `data.dat` file. This function should return a `float` with a value that the Black-Box process has evaluated.

# Task 5
Now we are interested in the minimum of the black box function. To obtain it we will perform an optimization procedure. By optimization we mean a process of selecting problem parameters that produce the best, with regard to a selected criterion, result.
In a general setting, the optimization problem consists of maximizing or minimizing a function by changing input parameters and systematically computing the value of the function.
There is a number of methods to realize this, here we will base our approach on approximating the function gradient using finite differences.

For this you will need to evaluate the gradient of the Black-Box process with respect to the value of parameters $x$ and $y$.

This can be achieved by probing the Black-Box function at $(x,y)$ and than at $(x+\delta,y)$ and$ (x,y+\delta)$ and constructing an approximation of the gradient $\nabla f$

Simple, first order, forward FD formulas give us:

$f_x= \frac{f(x +\delta ,y)- f(x ,y)} {\delta}$

$f_y= \frac{f(x, y +\delta )- f(x,y)} {\delta}$

and $\nabla f = [f_x, f_y]$.

Using the current position in the parametric space $(X,Y)$ we seek $(X_1, Y_1) = (X,Y) - \alpha \nabla f$.
I.e we will be moving towards decreasing values of $f$, and $\alpha$ is a value that makes sure we are not making to huge jumps, especially in regions that the gradient is large, that could be the case. For now start with $\alpha=0.1$.

# Task 6
Using functions created in Tasks 3 and 4 and the gradient evaluation procedure from task 5 formulate the optimization procedure and find the minimum. Change the way the process is started to `subprocess.popen`, add methods to monitor the execution of the process 


# Task 7
We will try to perform Bysian optimization (https://en.wikipedia.org/wiki/Bayesian_optimization) using the `pyGPGO` (https://pygpgo.readthedocs.io/en/latest/) module (if it is available).

In [ ]:
pip install pyGPGO

In [ ]:
import numpy as np
from pyGPGO.covfunc import squaredExponential
from pyGPGO.acquisition import Acquisition
from pyGPGO.surrogates.GaussianProcess import GaussianProcess
from pyGPGO.GPGO import GPGO

- `from pyGPGO.covfunc import squaredExponential`  
Imports the squared-exponential covariance function, which is the kernel of the Gaussian process will use to measure similarity between input points.
- `from pyGPGO.acquisition import Acquisition`  
  An acquisition function is the rule that decides where the optimiser should evaluate next by balancing “promising mean value” and “uncertainty.”
- `from pyGPGO.surrogates.GaussianProcess import GaussianProcess`
  Surrogate model class. In Bayesian optimisation, the surrogate is the cheap model that approximates the expensive objective evaluation.
- `from pyGPGO.GPGO import GPGO`  
  The main optimiser class that ties together the surrogate model, acquisition function, objective function, and parameter space.

In [ ]:
sexp = squaredExponential()
gp = GaussianProcess(sexp)
acq = Acquisition(mode='ExpectedImprovement')

- `sexp = squaredExponential()`  
  Creates the **kernel** (covariance function) for the Gaussian process. The squared-exponential kernel says that points closer together in `x` should usually have more similar function values.

- `gp = GaussianProcess(sexp)`  
  Creates the **Gaussian process surrogate model** using that kernel. This surrogate is the probabilistic model that approximates the objective function from the samples seen so far.

- `acq = Acquisition(mode='ExpectedImprovement')`  
  Creates the **acquisition function**. Expected Improvement (EI) is the rule that scores candidate points by combining the GP’s predicted value and uncertainty, so the optimiser balances exploration and exploitation.

In [ ]:
def f(x):
    '''objective function'''
    return (-x*x)

- `def f(x): return -x*x`  
  This is the **objective function** to optimise. The pyGPGO will **maximise**it.

In [ ]:
param = {'x': ('cont', [-1, 1])}

- `param = {'x': ('cont', [-1, 1])}`  
  Defines the **search space**: there is one parameter, `x`, it is **continuous**, and it is allowed to vary between `-1` and `1`.

In [ ]:
gpgo = GPGO(gp, acq, f, param)
gpgo.run(max_iter=0, resume=False)

- `gpgo = GPGO(gp, acq, f, param)`  
  Builds the **Bayesian optimization object** by combining the surrogate model (`gp`), acquisition function (`acq`), objective function (`f`), and parameter space (`param`).

- `gpgo.run(max_iter=0, resume=False)`  
  Starts a **fresh run**. Even with `max_iter=0`, pyGPGO still performs the initial random evaluations (`init_evals=3` by default), evaluates `f` there, and fits the first GP model. It does **not** perform any Bayesian optimization iterations yet.

In [ ]:
print('X:', gpgo.GP.X)
print('Y:', gpgo.GP.y)

In [ ]:
gpgo.run(max_iter=10, resume=True)

- `gpgo.run(max_iter=10, resume=True)`  
  **Continues** from the existing state instead of starting over. Here, pyGPGO uses the already collected initial samples and performs **10 Bayesian optimization iterations**, each time choosing a new point with the acquisition function and updating the GP with the new evaluation.

In [ ]:
print('X:', gpgo.GP.X)
print('Y:', gpgo.GP.y)

In [ ]:
x = np.linspace(-1,1,100)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(x, f(x))
plt.scatter(gpgo.GP.X,gpgo.GP.y)
plt.grid()

The BlackBox function used in this exercise evaluates: `double zval = (xval-45.68)*(xval+85.45) + (yval-58.84)*(yval-25.55);`  
Extend the pyGPGO snippet to:
1. Find the extremum using a simple Python function
2. Find the extremum by calling the BlackBox program   